In [ ]:
from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
THRESHOLDS_KB = (0, 5, 10, 20, 50)
N_PERM = 5000
SEED = 42
OUTDIR = Path("/content/bgc_proximity_sensitivity") if Path("/content").exists() else Path("bgc_proximity_sensitivity")
OUTDIR.mkdir(parents=True, exist_ok=True)

INPUTS = ("aor0-5.json", "aor0-5.gtf", "SM00066_ID_list.txt")
if any(not Path(x).exists() for x in INPUTS):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Place {INPUTS} in the working directory") from exc
    print("Upload the following three files:", ", ".join(INPUTS))
    files.upload()
for filename in INPUTS:
    if not Path(filename).is_file():
        raise FileNotFoundError(filename)

In [ ]:
# ===== 1. antiSMASH area intervals: convert 0-based half-open coordinates to 1-based closed coordinates =====
with open("aor0-5.json", encoding="utf-8") as fh:
    antismash = json.load(fh)
areas = []
for record in antismash["records"]:
    chromosome = record.get("id") or record.get("name")
    for i, area in enumerate(record.get("areas", []), start=1):
        if area.get("start") is None or area.get("end") is None:
            continue
        start, end = int(area["start"]) + 1, int(area["end"])
        if end < start:
            raise ValueError(f"Invalid area coordinates: {chromosome}:area{i}")
        areas.append((chromosome, start, end, f"{chromosome}:area{i}",
                      ",".join(map(str, area.get("products", [])))))
bgc = pd.DataFrame(areas, columns=["seqname", "start", "end", "bgc_id", "products"])
bgc = bgc.drop_duplicates(["seqname", "start", "end", "bgc_id"])

In [ ]:
# ===== 2. GTF: gene intervals for protein-coding genes with CDS features =====
cols = ["seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]
gtf = pd.read_csv("aor0-5.gtf", sep="\t", comment="#", names=cols,
                  usecols=["seqname", "feature", "start", "end", "attribute"],
                  dtype={"seqname": str, "attribute": str})
gtf["gene_id"] = gtf["attribute"].str.extract(r'gene_id "([^"]+)"', expand=False)
if gtf["gene_id"].isna().any():
    raise ValueError("gene_id could not be extracted from part of the GTF. Check the input format.")
cds_ids = set(gtf.loc[gtf["feature"].eq("CDS"), "gene_id"])
genes = (gtf.loc[gtf["gene_id"].isin(cds_ids)]
         .groupby(["seqname", "gene_id"], as_index=False, sort=True)
         .agg(start=("start", "min"), end=("end", "max")))
if genes["gene_id"].duplicated().any():
    raise ValueError("The same gene_id occurs on multiple chromosomes.")

with open("SM00066_ID_list.txt", encoding="utf-8") as fh:
    tf_ids = {line.strip() for line in fh if line.strip() and not line.startswith("#")}
missing = sorted(tf_ids - set(genes["gene_id"]))
if missing:
    raise ValueError(f"TF IDs absent from the GTF: {missing[:20]} (total: {len(missing)})")
genes["is_tf"] = genes["gene_id"].isin(tf_ids)
if len(genes) != 11902 or len(tf_ids) != 233 or len(bgc) != 80:
    print("Note: Counts differ from the existing analysis (11,902 genes, 233 TFs, and 80 areas).",
          f"Current counts: {len(genes)}, {len(tf_ids)}, {len(bgc)}")

In [ ]:
# ===== 3. Minimum interval distance to a BGC area on the same chromosome =====
genes["nearest_dist_bp"] = np.inf
genes["nearest_bgc_id"] = pd.Series([None] * len(genes), dtype="object")
for chrom, gsub in genes.groupby("seqname", sort=True):
    bsub = bgc.loc[bgc["seqname"].eq(chrom)]
    if bsub.empty:
        continue
    gs = gsub["start"].to_numpy(dtype=np.int64)[:, None]
    ge = gsub["end"].to_numpy(dtype=np.int64)[:, None]
    bs = bsub["start"].to_numpy(dtype=np.int64)[None, :]
    be = bsub["end"].to_numpy(dtype=np.int64)[None, :]
    # Return zero when a gene overlaps an area, matching interval_distance in the previous notebook.
    distances = np.maximum(np.maximum(bs - ge, gs - be), 0)
    argmin = distances.argmin(axis=1)
    genes.loc[gsub.index, "nearest_dist_bp"] = distances[np.arange(len(gsub)), argmin]
    genes.loc[gsub.index, "nearest_bgc_id"] = bsub["bgc_id"].to_numpy()[argmin]

tf = genes.loc[genes["is_tf"]].copy()
tf["nearest_dist_bp"] = tf["nearest_dist_bp"].replace(np.inf, np.nan)
tf.to_csv(OUTDIR / "tf_nearest_bgc_area.csv", index=False)

In [ ]:
# ===== 4. Permutation preserving the number of TFs per chromosome (same null hypothesis as the previous notebook) =====
thresholds_bp = np.array(THRESHOLDS_KB, dtype=np.int64) * 1000
observed = np.array([(tf["nearest_dist_bp"] <= t).sum() for t in thresholds_bp], dtype=int)
by_chrom = []
for chrom, sub in genes.groupby("seqname", sort=True):
    k = int(sub["is_tf"].sum())
    if k:
        by_chrom.append((chrom, sub["nearest_dist_bp"].to_numpy(), k))

rng = np.random.default_rng(SEED)
null_counts = np.zeros((N_PERM, len(thresholds_bp)), dtype=np.int16)
for i in range(N_PERM):
    for chrom, distances, k in by_chrom:
        chosen = rng.choice(len(distances), size=k, replace=False)
        null_counts[i] += (distances[chosen, None] <= thresholds_bp).sum(axis=0)

n_tf = len(tf)
rows = []
for j, kb in enumerate(THRESHOLDS_KB):
    r = null_counts[:, j].astype(int)
    obs = observed[j]
    expected = r.mean()
    lo, hi = np.percentile(r, [2.5, 97.5])
    # Include equality because the distribution is discrete. The two-sided p-value uses the absolute deviation from the null mean.
    p_upper = (np.count_nonzero(r >= obs) + 1) / (N_PERM + 1)
    p_lower = (np.count_nonzero(r <= obs) + 1) / (N_PERM + 1)
    p_two = (np.count_nonzero(abs(r - expected) >= abs(obs - expected)) + 1) / (N_PERM + 1)
    rows.append({
        "threshold_kb": kb,
        "n_tf": n_tf,
        "observed_count": obs,
        "observed_percent": 100 * obs / n_tf,
        "null_mean_count": expected,
        "null_mean_percent": 100 * expected / n_tf,
        "observed_minus_expected_count": obs - expected,
        "observed_minus_expected_percentage_points": 100 * (obs - expected) / n_tf,
        "observed_over_expected": obs / expected if expected else np.nan,
        "null_2_5_percentile_count": lo,
        "null_97_5_percentile_count": hi,
        "permutation_p_enrichment": p_upper,
        "permutation_p_depletion": p_lower,
        "permutation_p_two_sided": p_two,
        "n_permutations": N_PERM,
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUTDIR / "proximity_sensitivity_summary.csv", index=False)
pd.DataFrame(null_counts, columns=[f"{kb}kb" for kb in THRESHOLDS_KB]).to_csv(
    OUTDIR / "permutation_null_counts.csv", index=False)

In [ ]:
# ===== 5. Visualization: observed proportion and the 95% interval of the permutation null distribution =====
if plt is not None:
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    x = np.asarray(THRESHOLDS_KB, dtype=float)
    mean = summary["null_mean_percent"].to_numpy()
    lower = summary["null_2_5_percentile_count"].to_numpy() * 100 / n_tf
    upper = summary["null_97_5_percentile_count"].to_numpy() * 100 / n_tf
    ax.fill_between(x, lower, upper, color="#9bb7d3", alpha=0.4, label="Permutation null: 2.5–97.5%")
    ax.plot(x, mean, "o-", color="#32678e", label="Permutation mean")
    ax.plot(x, summary["observed_percent"], "s-", color="#bc4635", label="Observed Zn2Cys6 TFs")
    ax.set(xlabel="Distance from BGC area (kb)", ylabel="TFs near or inside BGC area (%)",
           xticks=x, xlim=(-1, max(x) + 2))
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTDIR / "proximity_sensitivity.png", dpi=300)
    fig.savefig(OUTDIR / "proximity_sensitivity.pdf")
    plt.close(fig)

print(f"BGC areas: {len(bgc)}; protein-coding genes: {len(genes)}; TFs: {n_tf}")
print(summary.round(4).to_string(index=False))
print("Results:", OUTDIR)

In [ ]:
# Download
if Path("/content").exists():
    archive = Path("/content/bgc_proximity_sensitivity.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in OUTDIR.iterdir():
            zf.write(path, arcname=path.name)
    print("Download archive:", archive)